In [21]:
from neo4j import GraphDatabase
from getpass import getpass
import anthropic, re, time, textwrap

URI  = "neo4j+s://c46c5d64.databases.neo4j.io"
USER = "c46c5d64"      # confirmed
DB   = "c46c5d64"      # confirmed

driver = GraphDatabase.driver(URI, auth=(USER, getpass("Neo4j password: ").strip()))
driver.verify_connectivity()
client = anthropic.Anthropic(api_key=getpass("Anthropic key: ").strip())

def cypher(q, **params):
    with driver.session(database=DB) as s:
        return [r.data() for r in s.run(q, **params)]

print("connected ·", cypher("MATCH (n) RETURN count(n) AS n")[0]["n"], "nodes")

Neo4j password: ··········
Anthropic key: ··········
connected · 65163 nodes


In [22]:
import re, time, textwrap

MODEL   = "claude-opus-5"
LIMIT_N = 50

SCHEMA = """
(:Patient {subject_id, gender, anchor_age})-[:HAS_ADMISSION]->
(:Admission {hadm_id, admittime, dischtime, admission_type})
(:Admission)-[:HAS_DIAGNOSIS]->(:Diagnosis {icd_code, long_title})
(:Admission)-[:HAS_PROCEDURE]->(:Procedure {icd_code, long_title})
(:Admission)-[:HAS_MEDICATION {route, dose, unit, starttime}]->(:Medication {name})
(:Admission)-[:INCLUDES_LAB]->(:LabEvent {label, valuenum, valueuom, charttime})

Medication names are lowercase. Lab labels are capitalised, e.g. 'Creatinine'.

Prefer aggregation (max, min, count, collect) over returning raw rows when the
question asks for a single value or a total. Only return raw rows when the
question asks to list or trace individual records.
"""


def llm(prompt, max_tokens=800):
    """Call the model and return its text. Raises with diagnostics if empty."""
    r = client.messages.create(
        model=MODEL, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}])
    text = "".join(b.text for b in r.content
                   if getattr(b, "type", "") == "text").strip()
    if not text:
        raise RuntimeError(
            f"no text returned — stop_reason={r.stop_reason}, "
            f"blocks={[getattr(b, 'type', '?') for b in r.content]}")
    return text


In [23]:
def route(question):
    out = llm(f"""You route clinical questions to one of two retrieval paths.

STRUCTURED — answerable from recorded fields: admissions, diagnoses, procedures,
  medication records, lab values. Questions of what, when, how many, in what order.
NARRATIVE — requires clinical reasoning, justification, or differential thinking
  recorded only in free-text notes. Questions of why, what was considered,
  what was ruled out.

Question: {question}

Reply with exactly one word: STRUCTURED or NARRATIVE.""", 200).upper()
    return "NARRATIVE" if "NARRATIVE" in out else "STRUCTURED"


In [24]:
def text2cypher(question, sid=None):
    q = llm(f"""You translate clinical questions into Neo4j Cypher.
{SCHEMA}
Question: {question}
{f"Scope to patient subject_id = {sid}." if sid else ""}
Return ONLY the Cypher query. No explanation, no markdown fences.
Always LIMIT results to at most {LIMIT_N} rows.""")
    return re.sub(r'^```(?:cypher)?|```$', '', q, flags=re.M).strip()


In [25]:
BANNED = re.compile(
    r'\b(CREATE|MERGE|SET|DELETE|DETACH|REMOVE|DROP|LOAD\s+CSV)\b', re.I)


def validate(q):
    if BANNED.search(q):
        return False, "rejected — write operation in generated query"
    if not re.search(r'\bLIMIT\b', q, re.I):
        q = q.rstrip().rstrip(';') + f"\nLIMIT {LIMIT_N};"
    return True, q


In [26]:
def ask(question, sid=None):
    print("QUESTION  " + question + "\n" + "─" * 66)

    path = route(question)
    print(f"[0] Router       → {path}\n")

    if path == "NARRATIVE":
        print("[B] Note retrieval\n"
              "    No structural mapping — no node or property holds clinical\n"
              "    reasoning. Routed to dense retrieval over discharge summaries.\n"
              "    Blocked: MIMIC-IV-Note requires PhysioNet credentialing.")
        return None

    t0 = time.time()
    q  = text2cypher(question, sid)
    print(f"[1] Text2Cypher  ({time.time()-t0:.1f}s)\n")
    print(textwrap.indent(q, "    ") + "\n")

    ok, q2 = validate(q)
    print(f"[2] Validation   {'✓ read-only, LIMIT enforced' if ok else '✗ ' + q2}\n")
    if not ok:
        return None

    t1   = time.time()
    rows = cypher(q2)
    print(f"[3] Neo4j        {len(rows)} row(s) in {time.time()-t1:.2f}s\n")

    if not rows:
        print("[3b] Fallback    Path A returned nothing → would route to note\n"
              "                 retrieval (blocked: MIMIC-IV-Note credentialing)")
        return None

    truncated = len(rows) >= LIMIT_N
    if truncated:
        print(f"[!] TRUNCATED    hit the {LIMIT_N}-row limit — this is a partial\n"
              f"                 result, not a complete one\n")

    print("[4] Answer\n")
    print(textwrap.indent(llm(
        f"Question: {question}\n\nRows from the clinical graph:\n{rows}\n\n"
        + (f"IMPORTANT: these rows were truncated at a {LIMIT_N}-row limit. "
           "Say so explicitly and do not present the list as complete.\n\n"
           if truncated else "")
        + "Answer in two or three sentences using only these rows. "
          "If the answer involves a sequence of events over time, state "
          "associations as temporal rather than causal. Otherwise answer plainly.",
        900), "    "))
    return None


In [28]:
ask("How many of her admissions record a leukaemia diagnosis?", sid=10014354)
ask("What was her highest recorded creatinine, and during which admission?", sid=10014354)

QUESTION  How many of her admissions record a leukaemia diagnosis?
──────────────────────────────────────────────────────────────────
[0] Router       → STRUCTURED

[1] Text2Cypher  (2.3s)

    MATCH (p:Patient {subject_id: 10014354})-[:HAS_ADMISSION]->(a:Admission)-[:HAS_DIAGNOSIS]->(d:Diagnosis)
    WHERE toLower(d.long_title) CONTAINS 'leukemia' OR toLower(d.long_title) CONTAINS 'leukaemia'
    RETURN count(DISTINCT a) AS leukaemia_admissions
    LIMIT 50

[2] Validation   ✓ read-only, LIMIT enforced

[3] Neo4j        1 row(s) in 0.04s

[4] Answer

    Based on the clinical graph data, 20 of her admissions record a leukaemia diagnosis. No further detail about the timing or context of these admissions is available from this result.
QUESTION  What was her highest recorded creatinine, and during which admission?
──────────────────────────────────────────────────────────────────
[0] Router       → STRUCTURED

[1] Text2Cypher  (3.5s)

    MATCH (p:Patient {subject_id: 10014354})-[:HAS_AD